In [ ]:
#command to mount my drive on colab

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!apt-get update -qq
!apt-get install -y openjdk-21-jdk-headless -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jdk-headless_21.0.13.1+2-1~24.04.1_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.13.1+2-1~24.04.1) over (21.0.12.1+1-1~24.04.4) ...
Preparing to unpack .../openjdk-21-jre-headless_21.0.13.1+2-1~24.04.1_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.13.1+2-1~24.04.1) over (21.0.12.1+1-1~24.04.4) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.13.1+2-1~24.04.1) ...
Installing new version of config file /etc/java-21-openjdk/security/blocked.certs ...
Installing new version of config file /etc/java-21-openjdk/security/default.policy ...
Installing new version of config file /etc/java-21-openjdk/security/java.security ...
Processing triggers for ca-certificates-java (20

In [2]:
#Setting/activating the environment variable as java (not compulsory everytime if you are on same runtime)
import os
os.environ["JAVA_HOME"]="/lib/jvm/java-21-openjdk-amd64"

In [3]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SQLContext
conf = SparkConf().set('spark.ui.port', '4050').setAppName("Housing").setMaster("local[2]")
sc = SparkContext.getOrCreate(conf=conf)
sqlContext = SQLContext(sc)

/usr/local/lib/python3.13/dist-packages/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [5]:
house_df = sqlContext.read.format('com.databricks.spark.csv').options(header='true', inferschema='true').load(r'Housing.csv')
house_df.printSchema()

root
 |-- price: integer (nullable = true)
 |-- area: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- stories: integer (nullable = true)
 |-- mainroad: string (nullable = true)
 |-- guestroom: string (nullable = true)
 |-- basement: string (nullable = true)
 |-- hotwaterheating: string (nullable = true)
 |-- airconditioning: string (nullable = true)
 |-- parking: integer (nullable = true)
 |-- prefarea: string (nullable = true)
 |-- furnishingstatus: string (nullable = true)



In [6]:
house_df.toPandas()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [7]:
house_df.dtypes

[('price', 'int'),
 ('area', 'int'),
 ('bedrooms', 'int'),
 ('bathrooms', 'int'),
 ('stories', 'int'),
 ('mainroad', 'string'),
 ('guestroom', 'string'),
 ('basement', 'string'),
 ('hotwaterheating', 'string'),
 ('airconditioning', 'string'),
 ('parking', 'int'),
 ('prefarea', 'string'),
 ('furnishingstatus', 'string')]

In [8]:
for i in house_df.dtypes:
  if i[1]=="string":
    print(i[0],end=" ")

mainroad guestroom basement hotwaterheating airconditioning prefarea furnishingstatus 

In [10]:
num_col=[i[0] for i in house_df.dtypes if i[1]=="int"]
num_col
cat_col=[i[0] for i in house_df.dtypes if i[1]!="int"]
cat_col

['mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'prefarea',
 'furnishingstatus']

In [11]:
from pyspark.ml.feature import VectorAssembler,StringIndexer,OneHotEncoder
stages=[]
for i in cat_col:
  string=StringIndexer(inputCol=i,outputCol=i+'Index')#An encoder defaults frequency encoding
  encoder=OneHotEncoder(inputCols=[string.getOutputCol()],outputCols=[i+"classVec"]) #Converting encoder column to OneHotEncoder
  stages+=[string,encoder]
assembler=[c+"classVec" for c in cat_col]+num_col

vector_assembler=VectorAssembler(inputCols=assembler,outputCol="features")
stages+=[vector_assembler]

In [12]:
from pyspark.ml import Pipeline
pipeline=Pipeline(stages=stages)
pipelinemodel=pipeline.fit(house_df)
df_1=pipelinemodel.transform(house_df)

In [13]:
df_1.show(2)

+--------+----+--------+---------+-------+--------+---------+--------+---------------+---------------+-------+--------+----------------+-------------+----------------+--------------+-----------------+-------------+----------------+--------------------+-----------------------+--------------------+-----------------------+-------------+----------------+---------------------+------------------------+--------------------+
|   price|area|bedrooms|bathrooms|stories|mainroad|guestroom|basement|hotwaterheating|airconditioning|parking|prefarea|furnishingstatus|mainroadIndex|mainroadclassVec|guestroomIndex|guestroomclassVec|basementIndex|basementclassVec|hotwaterheatingIndex|hotwaterheatingclassVec|airconditioningIndex|airconditioningclassVec|prefareaIndex|prefareaclassVec|furnishingstatusIndex|furnishingstatusclassVec|            features|
+--------+----+--------+---------+-------+--------+---------+--------+---------------+---------------+-------+--------+----------------+-------------+--------

In [14]:
select_col=['price','features']
df2=df_1.select(select_col)
df2.show(3)

+--------+--------------------+
|   price|            features|
+--------+--------------------+
|13300000|[1.0,1.0,1.0,1.0,...|
|12250000|[1.0,1.0,1.0,1.0,...|
|12250000|[1.0,1.0,0.0,1.0,...|
+--------+--------------------+
only showing top 3 rows


In [15]:
splits = df2.randomSplit([0.7, 0.3])
train_df = splits[0]
test_df = splits[1]
#train_df,test_df=vhouse_df.randomSplit([0.7,0.3])

In [16]:
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol = 'features', labelCol='price', maxIter=10)
lr_model = lr.fit(train_df)#
lr_predictions = lr_model.transform(test_df)
lr_predictions.select("prediction","price","features").show()
from pyspark.ml.evaluation import RegressionEvaluator
lr_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price",metricName="r2")
print("R Squared (R2) on test data = %g" % lr_evaluator.evaluate(lr_predictions))

+------------------+-------+--------------------+
|        prediction|  price|            features|
+------------------+-------+--------------------+
|         1750000.0|1750000|[0.0,1.0,1.0,1.0,...|
|         2100000.0|2100000|[1.0,1.0,1.0,1.0,...|
|2379999.9999999995|2380000|[1.0,1.0,1.0,1.0,...|
|2379999.9999999995|2380000|[1.0,1.0,1.0,1.0,...|
|         2380000.0|2380000|[1.0,1.0,1.0,1.0,...|
|2408000.0000000005|2408000|[0.0,1.0,1.0,1.0,...|
|2450000.0000000005|2450000|[0.0,0.0,1.0,1.0,...|
|         2450000.0|2450000|[1.0,1.0,1.0,1.0,...|
|         2450000.0|2450000|[1.0,1.0,1.0,1.0,...|
|2589999.9999999995|2590000|[1.0,1.0,0.0,1.0,...|
|         2590000.0|2590000|[1.0,1.0,1.0,1.0,...|
|2604000.0000000005|2604000|[0.0,1.0,1.0,1.0,...|
|2652999.9999999995|2653000|[1.0,1.0,1.0,1.0,...|
|2659999.9999999995|2660000|[1.0,1.0,1.0,1.0,...|
| 2835000.000000001|2835000|[0.0,1.0,1.0,0.0,...|
|2870000.0000000005|2870000|[0.0,1.0,1.0,1.0,...|
|         2870000.0|2870000|[1.0,1.0,1.0,1.0,...|


In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'price')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse = dt_evaluator.evaluate(dt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 1.22797e+06


In [ ]:
dt_evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")
r2 = dt_evaluator.evaluate(dt_predictions)
print("R2 on test data = %g" % r2)

R2 on test data = 0.539205
